In [1]:
import pandas as pd
import numpy as np
import wandb
import os

# ===============
## Code to use WANDB
# import wandb
# wandb.init(project="replearner")
# ===============


# Process data to look like P12 from RAINDROP orignal code

In [5]:
# =====================
# 1) 데이터 불러오기
# =====================
df = pd.read_feather('./data/mimic_data.feather')


# =====================
# 2) 변수 이름 리스트 생성
# =====================
param_list = sorted(df['itemid'].unique())
print(f"Total unique parameters: {len(param_list)}")
np.save('./processed_data/ts_params.npy', param_list)
print('ts_params.npy saved')

df_static = pd.read_feather('./data/mimic_data_static.feather')

static_dict = {}
for idx, row in df_static.iterrows():
    pid = row['pid']
    gender_onehot = row['gender']
    static_array = [row['age']] + [row['gender']] + [row['height']]
    static_dict[pid] = static_array

print('static_dict ready, example:', list(static_dict.items())[:3])

# 정적 변수 있음
static_param_list = ['age', 'gender', 'height']
d_static = 3
np.save('./processed_data/static_params.npy', static_param_list)

# =====================
# 3) 환자별 그룹화 및 PTdict_list 생성
# =====================
max_len = 577
max_offset = 48 * 60  # 48시간
F = len(param_list)

PTdict_list = []

for pid, group in df.groupby('pid'):
    group = group.sort_values(by='offset')
    ts = group[['offset', 'itemid', 'value']].values

    unq_offsets = []
    for sample in ts:
        offset = sample[0]
        if (offset not in unq_offsets) and (offset < max_offset):
            unq_offsets.append(offset)
    unq_offsets = np.array(unq_offsets)
    length = len(unq_offsets)

    # 시계열 및 타임스탬프 배열 생성
    Parr = np.zeros((max_len, F))
    Tarr = np.zeros((max_len, 1))

    for sample in ts:
        offset = sample[0]
        param = sample[1]
        value = sample[2]
        if offset < max_offset:
            try:
                time_id = np.where(offset == unq_offsets)[0][0]
                param_id = np.where(param_list == param)[0][0]
                Parr[time_id, param_id] = value
                Tarr[time_id, 0] = offset
            except:
                continue
    
    if pid in static_dict:
        static_array = static_dict[pid]
    else:
        static_array = [0] * d_static  # fallback

    my_dict = {
        'id': pid,
        'static': static_array,
        'extended_static': static_array,
        'arr': Parr,
        'time': Tarr,
        'length': length
    }
    PTdict_list.append(my_dict)

print(f"Total patients processed: {len(PTdict_list)}")

np.save('./processed_data/PTdict_list.npy', PTdict_list)
print('PTdict_list.npy saved, ready for Raindrop training')

# =====================
# 4) Outcomes 저장
# =====================

outcomes = pd.read_feather('./data/mimic_outcomes.feather') # AKI, CF, LoS, Mor
arr_outcomes = outcomes.values
np.save('./processed_data/arr_outcomes.npy', arr_outcomes)

Total unique parameters: 35
ts_params.npy saved
static_dict ready, example: [(23542772.0, [61.0, 1.0, 61.5]), (23864737.0, [61.0, 1.0, 67.0]), (29280967.0, [83.0, 0.0, 72.0])]
Total patients processed: 32350
PTdict_list.npy saved, ready for Raindrop training


In [ ]:
outcomes = pd.read_feather('./data/mimic_outcomes.feather')
arr_outcomes = outcomes.values
np.save('./processed_data/arr_outcomes.npy', arr_outcomes)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, average_precision_score
import random
import datetime


import wandb
wandb.init(project="replearner",
           config={
               'train_data': 'MIMIC',
               'valid_data': 'eICU',
               'architecture': 'Raindrop',
               'task': 'Mortality',
           }
           )
config = wandb.config

# ======== 1) 데이터 로드 ========
# Watch out f[]'=p
PTdict_list = np.load('./processed_data/PTdict_list.npy', allow_pickle=True)
arr_outcomes = np.load('./processed_data/arr_outcomes.npy', allow_pickle=True)
y = arr_outcomes[:, 3].reshape(-1, 1)  # # AKI, CF, LoS, Mor, we go with Mortality first

# ======== 2) 데이터 split ========
idx = np.arange(len(PTdict_list))
np.random.seed(42)
np.random.shuffle(idx)

n = len(idx)
train_idx = idx[:int(0.8*n)]
val_idx = idx[int(0.8*n):int(0.9*n)]
test_idx = idx[int(0.9*n):]

Ptrain = PTdict_list[train_idx]
Pval = PTdict_list[val_idx]
Ptest = PTdict_list[test_idx]

ytrain = y[train_idx]
yval = y[val_idx]
ytest = y[test_idx]

# ======== 3) Normalization ========
from utils_rd import getStats, getStats_static, tensorize_normalize

T, F = Ptrain[0]['arr'].shape
D = len(Ptrain[0]['extended_static'])

Ptrain_tensor_raw = np.zeros((len(Ptrain), T, F))
Ptrain_static_tensor_raw = np.zeros((len(Ptrain), D))
for i in range(len(Ptrain)):
    Ptrain_tensor_raw[i] = Ptrain[i]['arr']
    Ptrain_static_tensor_raw[i] = Ptrain[i]['extended_static']

mf, stdf = getStats(Ptrain_tensor_raw)
ms, ss = getStats_static(Ptrain_static_tensor_raw, dataset='Default')

Ptrain_tensor, Ptrain_static_tensor, Ptrain_time_tensor, ytrain_tensor = tensorize_normalize(
    Ptrain, ytrain, mf, stdf, ms, ss)
Pval_tensor, Pval_static_tensor, Pval_time_tensor, yval_tensor = tensorize_normalize(
    Pval, yval, mf, stdf, ms, ss)
Ptest_tensor, Ptest_static_tensor, Ptest_time_tensor, ytest_tensor = tensorize_normalize(
    Ptest, ytest, mf, stdf, ms, ss)

# Permute to [T, B, F]
Ptrain_tensor = Ptrain_tensor.permute(1, 0, 2).cuda()
Pval_tensor = Pval_tensor.permute(1, 0, 2).cuda()
Ptest_tensor = Ptest_tensor.permute(1, 0, 2).cuda()

Ptrain_time_tensor = Ptrain_time_tensor.squeeze(2).permute(1, 0).cuda()
Pval_time_tensor = Pval_time_tensor.squeeze(2).permute(1, 0).cuda()
Ptest_time_tensor = Ptest_time_tensor.squeeze(2).permute(1, 0).cuda()

Ptrain_static_tensor = Ptrain_static_tensor.cuda()
Pval_static_tensor = Pval_static_tensor.cuda()
Ptest_static_tensor = Ptest_static_tensor.cuda()

ytrain_tensor = ytrain_tensor.cuda()
yval_tensor = yval_tensor.cuda()
ytest_tensor = ytest_tensor.cuda()

# ======== 4) 모델 생성 ========
from models_rd import Raindrop_Mod

d_inp = F
d_ob = 4
d_model = d_inp * d_ob
d_static = D
max_len = T
n_classes = 2

global_structure = torch.ones(d_inp, d_inp).cuda()

model = Raindrop_Mod(
    d_inp=d_inp, d_model=d_model, nhead=2, nhid=2*d_model,
    nlayers=2, dropout=0.2, max_len=max_len, d_static=d_static,
    MAX=100, aggreg='mean', n_classes=n_classes,
    global_structure=global_structure
).cuda()

optimizer = optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

# ======== 5) 학습 ========
num_epochs = 5
batch_size = 16

for epoch in range(num_epochs):
    epoch_start = datetime.datetime.now()
    model.train()
    indices = np.arange(Ptrain_tensor.shape[1])
    np.random.shuffle(indices)

    train_losses = []
    all_train_preds = []
    all_train_probs = []
    all_train_labels = []

    for i in range(0, len(indices), batch_size):
        idx = indices[i:i+batch_size]
        P_batch = Ptrain_tensor[:, idx, :]
        Ptime_batch = Ptrain_time_tensor[:, idx]
        Pstatic_batch = Ptrain_static_tensor[idx]
        y_batch = ytrain_tensor[idx]

        lengths = torch.sum(Ptime_batch > 0, dim=0)

        outputs, _, _ = model(P_batch, Pstatic_batch, Ptime_batch, lengths)
        loss = criterion(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
        preds = np.argmax(probs, axis=1)
        labels = y_batch.cpu().numpy()

        all_train_preds.extend(preds)
        all_train_probs.extend(probs[:, 1])
        all_train_labels.extend(labels)

    train_acc = accuracy_score(all_train_labels, all_train_preds)
    train_auroc = roc_auc_score(all_train_labels, all_train_probs)
    train_auprc = average_precision_score(all_train_labels, all_train_probs)

    # ======== 6) Validation ========
    model.eval()
    with torch.no_grad():
        lengths_val = torch.sum(Pval_time_tensor > 0, dim=0)
        outputs_val, _, _ = model(Pval_tensor, Pval_static_tensor, Pval_time_tensor, lengths_val)
        probs_val = torch.softmax(outputs_val, dim=1).cpu().numpy()
        preds_val = np.argmax(probs_val, axis=1)
        y_val_true = yval_tensor.cpu().numpy().reshape(-1)

        valid_loss = criterion(outputs_val, yval_tensor).item()
        valid_acc = accuracy_score(y_val_true, preds_val)
        valid_auroc = roc_auc_score(y_val_true, probs_val[:, 1])
        valid_auprc = average_precision_score(y_val_true, probs_val[:, 1])
    
    epoch_time = datetime.datetime.now() - epoch_start

    # ======== 7) wandb logging ========
    wandb.log({
        "train_loss": np.mean(train_losses),
        "valid_loss": valid_loss,
        "train_acc": train_acc,
        "train_auroc": train_auroc,
        "train_auprc": train_auprc,
        "valid_acc": valid_acc,
        "valid_auroc": valid_auroc,
        "valid_auprc": valid_auprc,
        "epoch_time_sec": epoch_time.total_seconds() // 60.0,
    }, step=epoch)
    print(f"Epoch {0} finished, at time {datetime.datetime.now()}")

# ======== 8) Test Evaluation ========
model.eval()
with torch.no_grad():
    lengths_test = torch.sum(Ptest_time_tensor > 0, dim=0)
    outputs_test, _, embs_test = model(Ptest_tensor, Ptest_static_tensor, Ptest_time_tensor, lengths_test)
    probs_test = torch.softmax(outputs_test, dim=1).cpu().numpy()
    preds_test = np.argmax(probs_test, axis=1)
    y_test_true = ytest_tensor.cpu().numpy().reshape(-1)

    test_loss = criterion(outputs_test, ytest_tensor).item()
    acc_test = accuracy_score(y_test_true, preds_test)
    auc_test = roc_auc_score(y_test_true, probs_test[:, 1])
    auprc_test = average_precision_score(y_test_true, probs_test[:, 1])

# wandb logging
wandb.log({
    "test_loss": test_loss,
    "test_acc": acc_test,
    "test_auroc": auc_test,
    "test_auprc": auprc_test
})

train_acc,▁▃▃▃▃▄▄▅▅▆▇█
train_auprc,▁▃▃▄▄▄▅▅▆▆▇█
train_auroc,▁▄▄▄▅▅▅▆▆▇▇█
train_loss,█▆▆▅▅▅▄▄▃▃▂▁
valid_acc,▄▄▄▁█▆▆█▃▁▁▃
valid_auprc,▅▇█████▇▄▄▂▁
valid_auroc,▅▇▇███▇▇▆▄▁▁
valid_loss,▂▁▁▁▁▁▃▂▂▆▅█
train_acc,0.84502
train_auprc,0.72704
train_auroc,0.88194


c:\Users\Jaewon\Documents\imvts_rep\.conda\Lib\site-packages\torch\nn\modules\transformer.py:375: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
c:\Users\Jaewon\Documents\imvts_rep\models_rd.py:506: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen/native/IndexingUtils.h:30.)
  adj[torch.eye(self.d_inp).byte()] = 1
c:\Users\Jaewon\Documents\imvts_rep\models_rd.py:506: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen/native/IndexingUtils.h:30.)
  adj[torch.eye(self.d_inp).byte()] = 1


Epoch 0 finished, at time 2025-07-10 16:30:29.708175


c:\Users\Jaewon\Documents\imvts_rep\models_rd.py:506: UserWarning: indexing with dtype torch.uint8 is now deprecated, please use a dtype torch.bool instead. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen/native/IndexingUtils.h:30.)
  adj[torch.eye(self.d_inp).byte()] = 1


KeyboardInterrupt: 

In [1]:
import pandas as pd

data = pd.read_feather('./rd_results/predictions_train_mimic_mor_32.feather')